Background:
Thunderbolt Charge, Inc (TCI) is a fast-growing electric vehicle (EV) charging network
operator focused on building high-speed charging infrastructure across the American
Southwest and Pacific Northwest. As demand for EVs increases, so does the need for well-
placed charging hubs that provide coverage, reliability, and convenience to EV drivers.
To guide its regional expansion, TCI is evaluating where to locate new fast-charging hubs
across key cities in California, Arizona, Nevada, Utah, Oregon, and Idaho. Your team has
been brought in to help the company determine how to strategically place hubs to best
serve its current and future customers.
The company has compiled a list of 22 candidate cities, along with:
- Pairwise distances between each city (in miles),
- Estimated number of EVs registered in each metro area.
TCI is considering multiple service models. Your job is to explore these different scenarios
and recommend locations for charging hubs that achieve the business goals in each case. All
analysis must be done with linear models; non-linear approaches will not be accepted.
Decision Problems
1. Minimize the Worst-Case Distance (Max Distance Minimization)
TCI can initially build only 4 charging hubs. To ensure all drivers are reasonably close to a
charging location, your goal is to minimize the maximum distance any city is from its
assigned charging hub. In which cities should the 4 hubs be located to achieve this?
2. Guarantee Access Within 200 Miles
TCI wants to ensure that all EV drivers are within 200 miles of at least one hub, regardless
of where they live in the region. What is the minimum number of hubs needed to achieve
this? In which cities should these hubs be placed?
3. Plan for Redundant Coverage
Occasionally, a hub may be offline for maintenance or experience high traffic. To ensure
redundancy and reliability, the company wants to guarantee that every city is within 200
miles of at least two different hubs. What is the minimum number of hubs needed to ensure
this dual coverage? Where should those hubs be located?
4. Maximize Population Coverage
Budget constraints allow the company to build only 3 charging hubs at this stage.
Thunderbolt Charge wants to maximize the number of EVs served, i.e., the total number of
EVs that are within 200 miles of at least one of the hubs. Which 3 cities should be selected
as hub locations to maximize total EV coverage?

In [ ]:
import gurobipy as gp
from gurobipy import GRB
from typing import Dict, List, Tuple

In [ ]:
pip install gurobipy

In [ ]:
## Cities

CITIES: List[str] = [
    'San Diego, CA (Downtown)',
    'San Diego, CA (North County)',
    'Los Angeles, CA (Downtown)',
    'Los Angeles, CA (San Fernando Valley)',
    'San Jose, CA',
    'San Francisco, CA',
    'Sacramento, CA',
    'Fresno, CA',
    'Palm Springs, CA',
    'Phoenix, AZ (Downtown)',
    'Phoenix, AZ (Tempe)',
    'Tucson, AZ',
    'Flagstaff, AZ',
    'Las Vegas, NV',
    'Reno, NV',
    'Salt Lake City, UT',
    'Provo, UT',
    'St. George, UT',
    'Portland, OR',
    'Eugene, OR',
    'Hood River, OR',
    'Boise, ID'
]

In [ ]:
# EV populations (per city)
EV_POP: Dict[str, float] = {
    'San Diego, CA (Downtown)': 50000.0,
    'San Diego, CA (North County)': 45000.0,
    'Los Angeles, CA (Downtown)': 120000.0,
    'Los Angeles, CA (San Fernando Valley)': 110000.0,
    'San Jose, CA': 90000.0,
    'San Francisco, CA': 85000.0,
    'Sacramento, CA': 60000.0,
    'Fresno, CA': 15000.0,
    'Palm Springs, CA': 10000.0,
    'Phoenix, AZ (Downtown)': 40000.0,
    'Phoenix, AZ (Tempe)': 35000.0,
    'Tucson, AZ': 25000.0,
    'Flagstaff, AZ': 5000.0,
    'Las Vegas, NV': 30000.0,
    'Reno, NV': 15000.0,
    'Salt Lake City, UT': 20000.0,
    'Provo, UT': 10000.0,
    'St. George, UT': 5000.0,
    'Portland, OR': 60000.0,
    'Eugene, OR': 25000.0,
    'Hood River, OR': 5000.0,
    'Boise, ID': 20000.0
}

In [ ]:
# Distance matrix (rows/cols follow the exact order in CITIES)
DIST_MATRIX: List[List[float]] = [
    [0.0, 32.0, 111.0, 131.0, 416.0, 458.0, 472.0, 315.0, 85.0, 299.0, 313.0, 363.0, 360.0, 265.0, 493.0, 627.0, 602.0, 364.0, 930.0, 845.0, 926.0, 754.0],
    [32.0, 0.0, 80.0, 100.0, 386.0, 428.0, 440.0, 283.0, 66.0, 306.0, 320.0, 372.0, 355.0, 243.0, 460.0, 605.0, 581.0, 346.0, 898.0, 813.0, 894.0, 724.0],
    [111.0, 80.0, 0.0, 20.0, 305.0, 347.0, 361.0, 205.0, 99.0, 358.0, 372.0, 440.0, 384.0, 229.0, 388.0, 579.0, 560.0, 337.0, 824.0, 737.0, 822.0, 669.0],
    [131.0, 100.0, 20.0, 0.0, 286.0, 328.0, 342.0, 186.0, 115.0, 373.0, 387.0, 457.0, 394.0, 230.0, 371.0, 576.0, 558.0, 339.0, 807.0, 719.0, 806.0, 657.0],
    [416.0, 386.0, 305.0, 286.0, 0.0, 42.0, 88.0, 123.0, 386.0, 615.0, 628.0, 713.0, 590.0, 383.0, 188.0, 587.0, 587.0, 459.0, 565.0, 468.0, 578.0, 526.0],
    [458.0, 428.0, 347.0, 328.0, 42.0, 0.0, 75.0, 162.0, 442.0, 654.0, 667.0, 757.0, 625.0, 418.0, 186.0, 601.0, 601.0, 481.0, 534.0, 435.0, 549.0, 518.0],
    [472.0, 440.0, 361.0, 342.0, 88.0, 75.0, 0.0, 158.0, 442.0, 635.0, 648.0, 737.0, 625.0, 417.0, 112.0, 601.0, 601.0, 445.0, 482.0, 387.0, 492.0, 444.0],
    [315.0, 283.0, 205.0, 186.0, 123.0, 162.0, 158.0, 0.0, 272.0, 492.0, 506.0, 591.0, 468.0, 262.0, 193.0, 509.0, 503.0, 345.0, 623.0, 534.0, 625.0, 511.0],
    [85.0, 66.0, 99.0, 115.0, 386.0, 442.0, 429.0, 272.0, 0.0, 259.0, 273.0, 343.0, 295.0, 180.0, 430.0, 542.0, 518.0, 265.0, 869.0, 788.0, 861.0, 675.0],
    [299.0, 306.0, 358.0, 373.0, 615.0, 654.0, 635.0, 492.0, 259.0, 0.0, 14.0, 106.0, 123.0, 256.0, 601.0, 504.0, 469.0, 265.0, 1005.0, 941.0, 983.0, 736.0],
    [313.0, 320.0, 372.0, 387.0, 628.0, 667.0, 648.0, 506.0, 273.0, 14.0, 0.0, 96.0, 109.0, 262.0, 612.0, 507.0, 470.0, 272.0, 992.0, 938.0, 982.0, 742.0],
    [363.0, 372.0, 440.0, 457.0, 713.0, 757.0, 737.0, 591.0, 343.0, 106.0, 96.0, 0.0, 209.0, 362.0, 706.0, 591.0, 584.0, 387.0, 1110.0, 1014.0, 1088.0, 836.0],
    [360.0, 355.0, 384.0, 394.0, 590.0, 625.0, 593.0, 468.0, 295.0, 123.0, 109.0, 209.0, 0.0, 207.0, 539.0, 384.0, 347.0, 169.0, 917.0, 862.0, 891.0, 629.0],
    [265.0, 243.0, 229.0, 230.0, 383.0, 418.0, 387.0, 262.0, 180.0, 256.0, 262.0, 362.0, 207.0, 0.0, 345.0, 362.0, 338.0, 108.0, 755.0, 687.0, 737.0, 517.0],
    [493.0, 460.0, 388.0, 371.0, 188.0, 186.0, 112.0, 193.0, 430.0, 601.0, 612.0, 706.0, 539.0, 345.0, 0.0, 428.0, 428.0, 379.0, 775.0, 687.0, 737.0, 517.0],
    [627.0, 605.0, 579.0, 576.0, 587.0, 601.0, 601.0, 509.0, 542.0, 504.0, 507.0, 591.0, 384.0, 362.0, 428.0, 0.0, 38.0, 268.0, 635.0, 616.0, 593.0, 296.0],
    [602.0, 581.0, 560.0, 558.0, 587.0, 601.0, 601.0, 503.0, 518.0, 469.0, 470.0, 584.0, 347.0, 338.0, 428.0, 38.0, 0.0, 240.0, 666.0, 643.0, 626.0, 331.0],
    [364.0, 346.0, 337.0, 339.0, 459.0, 481.0, 445.0, 345.0, 265.0, 265.0, 272.0, 387.0, 169.0, 108.0, 379.0, 268.0, 240.0, 0.0, 748.0, 693.0, 723.0, 471.0],
    [930.0, 898.0, 824.0, 807.0, 565.0, 534.0, 482.0, 623.0, 869.0, 1005.0, 992.0, 1110.0, 917.0, 755.0, 775.0, 635.0, 666.0, 748.0, 0.0, 102.0, 58.0, 345.0],
    [845.0, 813.0, 737.0, 719.0, 468.0, 435.0, 387.0, 534.0, 788.0, 941.0, 938.0, 1014.0, 862.0, 687.0, 687.0, 616.0, 643.0, 693.0, 102.0, 0.0, 138.0, 345.0],
    [926.0, 894.0, 822.0, 806.0, 578.0, 549.0, 492.0, 625.0, 861.0, 983.0, 982.0, 1088.0, 891.0, 737.0, 737.0, 593.0, 626.0, 723.0, 58.0, 138.0, 0.0, 289.0],
    [754.0, 724.0, 669.0, 657.0, 526.0, 518.0, 444.0, 511.0, 675.0, 736.0, 742.0, 836.0, 629.0, 517.0, 517.0, 296.0, 331.0, 471.0, 345.0, 345.0, 289.0, 0.0],
]


In [ ]:
# Convert DIST_MATRIX into a distance dictionary d[(i,j)]
DIST: Dict[Tuple[str, str], float] = {}
for i, city_i in enumerate(CITIES):
    for j, city_j in enumerate(CITIES):
        DIST[(city_i, city_j)] = float(DIST_MATRIX[i][j])


def coverage_indicator(radius: float) -> Dict[Tuple[str, str], int]:
    """Return a[(i,j)] = 1 if DIST(i,j) <= radius else 0."""
    a = {}
    for i in CITIES:
        for j in CITIES:
            a[(i, j)] = 1 if DIST[(i, j)] <= radius else 0
    return a

# 1) Part 1 Minimize Worst Case Distance

In [ ]:
def part1(p: int = 4, verbose: bool = False):
    """
    Choose exactly p hubs and assign each city to one hub so that the maximum
    assigned distance is minimized.

    Variables:
      y[j] in {0,1}: hub at city j?
      x[i,j] in {0,1}: assign city i to hub j?
      Dmax >= 0: the worst-case (maximum) assigned distance

    Objective:
      minimize Dmax

    Constraints:
      sum_j y[j] = p
      sum_j x[i,j] = 1 for all i
      x[i,j] <= y[j] for all i,j
      sum_j DIST[i,j] * x[i,j] <= Dmax for all i
    """
    m = gp.Model("Part1_p_center")
    m.Params.OutputFlag = 1 if verbose else 0

    y = m.addVars(CITIES, vtype=GRB.BINARY, name="y")
    x = m.addVars(CITIES, CITIES, vtype=GRB.BINARY, name="x")
    Dmax = m.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="Dmax")

    m.setObjective(Dmax, GRB.MINIMIZE)

    m.addConstr(gp.quicksum(y[j] for j in CITIES) == p, name="hub_count")

    for i in CITIES:
        m.addConstr(gp.quicksum(x[i, j] for j in CITIES) == 1, name=f"assign_once[{i}]")

    for i in CITIES:
        for j in CITIES:
            m.addConstr(x[i, j] <= y[j], name=f"link_open[{i},{j}]")

    for i in CITIES:
        m.addConstr(
            gp.quicksum(DIST[(i, j)] * x[i, j] for j in CITIES) <= Dmax,
            name=f"worst_case_def[{i}]"
        )

    m.optimize()

    if m.Status != GRB.OPTIMAL:
        raise RuntimeError(f"Part 1 failed. Gurobi status={m.Status}")

    hubs = [j for j in CITIES if y[j].X > 0.5]
    assignment = {}
    for i in CITIES:
        for j in CITIES:
            if x[i, j].X > 0.5:
                assignment[i] = j
                break

    return Dmax.X, hubs, assignment


# Part 2 — Minimum Hubs withtin 200 miles

In [ ]:
def part2(radius: float = 200.0, verbose: bool = False):
    """
    Minimum number of hubs so every city is within 'radius' miles of at least one hub.

    Variables:
      y[j] in {0,1}: hub at city j?

    Constraints:
      for each i: sum_j a[i,j] * y[j] >= 1

    Objective:
      minimize sum_j y[j]
    """
    a = coverage_indicator(radius)

    m = gp.Model("Part2_set_cover")
    m.Params.OutputFlag = 1 if verbose else 0

    y = m.addVars(CITIES, vtype=GRB.BINARY, name="y")
    m.setObjective(gp.quicksum(y[j] for j in CITIES), GRB.MINIMIZE)

    for i in CITIES:
        m.addConstr(
            gp.quicksum(a[(i, j)] * y[j] for j in CITIES) >= 1,
            name=f"cover[{i}]"
        )

    m.optimize()

    if m.Status != GRB.OPTIMAL:
        raise RuntimeError(f"Part 2 failed. Gurobi status={m.Status}")

    hubs = [j for j in CITIES if y[j].X > 0.5]
    return len(hubs), hubs


# 3) Part 3 — Redundency Coverage

In [ ]:
def part3(radius: float = 200.0, k: int = 2, verbose: bool = False):
    """
    Minimum number of hubs so every city is within 'radius' miles of at least k hubs.

    Variables:
      y[j] in {0,1}: hub at city j?

    Constraints:
      for each i: sum_j a[i,j] * y[j] >= k

    Objective:
      minimize sum_j y[j]

    Note:
      This can be INFEASIBLE for some radii if any city has fewer than k candidate hubs within radius.
    """
    a = coverage_indicator(radius)

    # Quick pre-check: if any city has <k possible covering hubs, infeasible is guaranteed.
    for i in CITIES:
        candidates = sum(a[(i, j)] for j in CITIES)
        if candidates < k:
            raise RuntimeError(
                f"Part 3 is infeasible at radius={radius}: '{i}' has only {candidates} candidate hub(s) within radius."
            )

    m = gp.Model("Part3_dual_cover")
    m.Params.OutputFlag = 1 if verbose else 0

    y = m.addVars(CITIES, vtype=GRB.BINARY, name="y")
    m.setObjective(gp.quicksum(y[j] for j in CITIES), GRB.MINIMIZE)

    for i in CITIES:
        m.addConstr(
            gp.quicksum(a[(i, j)] * y[j] for j in CITIES) >= k,
            name=f"cover_k[{i}]"
        )

    m.optimize()

    if m.Status != GRB.OPTIMAL:
        raise RuntimeError(f"Part 3 failed. Gurobi status={m.Status}")

    hubs = [j for j in CITIES if y[j].X > 0.5]
    return len(hubs), hubs


# 4) Part 4 — Maximize EV coverage with 3 hubs

In [ ]:
def part4(radius: float = 200.0, p: int = 3, verbose: bool = False):
    """
    Choose exactly p hubs to maximize total EV population covered within radius.

    Variables:
      y[j] in {0,1}: hub at city j?
      z[i] in {0,1}: is city i covered by at least one chosen hub?

    Constraints:
      sum_j y[j] = p
      z[i] <= sum_j a[i,j] * y[j]   (if no hub covers i, z[i] must be 0)

    Objective:
      maximize sum_i EV_POP[i] * z[i]
    """
    a = coverage_indicator(radius)

    m = gp.Model("Part4_max_coverage")
    m.Params.OutputFlag = 1 if verbose else 0

    y = m.addVars(CITIES, vtype=GRB.BINARY, name="y")
    z = m.addVars(CITIES, vtype=GRB.BINARY, name="z")

    m.setObjective(gp.quicksum(EV_POP[i] * z[i] for i in CITIES), GRB.MAXIMIZE)

    m.addConstr(gp.quicksum(y[j] for j in CITIES) == p, name="hub_count")

    for i in CITIES:
        m.addConstr(
            z[i] <= gp.quicksum(a[(i, j)] * y[j] for j in CITIES),
            name=f"link_cover[{i}]"
        )

    m.optimize()

    if m.Status != GRB.OPTIMAL:
        raise RuntimeError(f"Part 4 failed. Gurobi status={m.Status}")

    hubs = [j for j in CITIES if y[j].X > 0.5]
    covered_cities = [i for i in CITIES if z[i].X > 0.5]
    total_ev = m.ObjVal

    return total_ev, hubs, covered_cities

In [ ]:
def main():
    R = 200.0

    print(f"Cities loaded: {len(CITIES)}")

    # Part 1
    Dmax, hubs1, assign1 = part1(p=4, verbose=False)
    print("\nPart 1 — Minimize worst-case distance with 4 hubs")
    print(f"  Optimal Dmax: {Dmax:.2f} miles")
    print(f"  Hubs ({len(hubs1)}): {hubs1}")

    # Optional: show worst-case city
    worst_city = None
    worst_dist = -1.0
    for i in CITIES:
        j = assign1[i]
        dist_ij = DIST[(i, j)]
        if dist_ij > worst_dist:
            worst_dist = dist_ij
            worst_city = i
    print(f"  Worst-served city: {worst_city} at {worst_dist:.0f} miles")

    # Part 2
    k2, hubs2 = part2(radius=R, verbose=False)
    print(f"\nPart 2 — Min hubs so everyone within {R:.0f} miles of ≥1 hub")
    print(f"  Minimum hubs: {k2}")
    print(f"  Hubs: {hubs2}")

    # Part 3
    print(f"\nPart 3 — Min hubs so everyone within {R:.0f} miles of ≥2 hubs (redundant coverage)")
    try:
        k3, hubs3 = part3(radius=R, k=2, verbose=False)
        print(f"  Minimum hubs: {k3}")
        print(f"  Hubs: {hubs3}")
    except RuntimeError as e:
        print(f"  {e}")

    # Part 4
    total_ev, hubs4, covered4 = part4(radius=R, p=3, verbose=False)
    print(f"\nPart 4 — Max EV population covered within {R:.0f} miles using 3 hubs")
    print(f"  Total EV covered: {total_ev:,.0f}")
    print(f"  Hubs ({len(hubs4)}): {hubs4}")
    print(f"  # Cities covered: {len(covered4)}/{len(CITIES)}")
    print(f"  Covered cities: {covered4}")


if __name__ == "__main__":
    main()